# Kiểm tra phân bố ngày TopCV

Notebook này phân tích CSV đã crawl và metadata URL trong sitemap Wayback. `posted_date` của CSV hiện có **không ghi nguồn cho từng dòng**: crawler cũ ưu tiên `datePosted` trong JSON-LD, rồi gán `<lastmod>` của sitemap nếu thiếu. Vì vậy biểu đồ phản ánh **ngày đã ghi trong CSV**, chưa đủ để gọi mọi điểm là ngày đăng xác thực.

Crawler mới không dùng `lastmod` để điền ngày đăng. CSV hiện có không được thay đổi tự động; muốn biết chính xác nguồn ngày của từng tin cũ cần crawl lại hoặc đối chiếu HTML gốc.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src/crawler/topcv_crawler.py").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Mở notebook trong thư mục dự án hoặc notebooks/")
raw_dir = project_root / "data/raw"
job_path = raw_dir / "topcv_it_jobs_wayback.csv"
url_path = raw_dir / "topcv_it_job_urls.csv"
if not job_path.exists() or not url_path.exists():
    raise FileNotFoundError("Cần cả CSV chi tiết và CSV URL TopCV")

jobs_raw = pd.read_csv(job_path, dtype=str, keep_default_na=False)
urls = pd.read_csv(url_path, dtype=str, keep_default_na=False)
if "url" not in jobs_raw or "posted_date" not in jobs_raw or "url" not in urls:
    raise ValueError("CSV thiếu cột url hoặc posted_date")

# Một URL chỉ đếm một tin; kiểm tra số dòng bị loại ở cell sau.
jobs = jobs_raw.drop_duplicates(subset=["url"], keep="first").copy()
jobs["posted_dt"] = pd.to_datetime(jobs["posted_date"], errors="coerce")
print(f"CSV chi tiết: {len(jobs_raw):,} dòng; {len(jobs):,} URL độc bản")
print(f"Trùng URL: {len(jobs_raw) - len(jobs):,}; ngày không đọc được: {jobs.posted_dt.isna().sum():,}")


## 1. Kiểm tra nguồn ngày và mốc Wayback

`year` và `snapshot_period` trong CSV URL là **mốc sitemap được yêu cầu**. Chúng không xác nhận ngày đăng tin hay thời điểm thực Wayback trả về. Crawler cũ lấy sitemap của vài mốc cố định, giữ URL ở mốc đầu tiên tìm thấy, rồi tải trang chi tiết từ TopCV hiện tại. Trang chi tiết của tin cũ chưa được tải từ Wayback.

URL tin có thể đổi slug hoặc chuyển từ trang brand sang trang việc làm; khi URL không khớp, notebook ghép theo ID tin nếu ID đó chỉ xuất hiện một lần trong CSV sitemap.

Nếu `posted_date` trùng `<lastmod>`, ta **không thể kết luận** crawler đã dùng fallback; JSON-LD cũng có thể chứa cùng ngày. Cột `sitemap_lastmod` là tên mới của `posted_date_fallback` trong CSV URL cũ. Theo [đặc tả Sitemap](https://www.sitemaps.org/protocol.html), `<lastmod>` là ngày sửa trang.


In [ ]:
import re

# URL TopCV có thể đổi slug hoặc chuyển từ /brand/... sang /viec-lam/...
# Trong trường hợp đó chỉ ghép ID khi ID là duy nhất trong CSV sitemap.
def job_id(url):
    match = re.search(r"(?:/|-)(?:j)?(\d{5,})\.html$", url)
    return match.group(1) if match else ""

lastmod_col = "sitemap_lastmod" if "sitemap_lastmod" in urls.columns else "posted_date_fallback"
if lastmod_col not in urls:
    raise ValueError("CSV URL thiếu ngày lastmod của sitemap")
metadata = urls[["url", lastmod_col, "year", "snapshot_period"]].rename(
    columns={lastmod_col: "sitemap_lastmod", "year": "snapshot_year"}
)
metadata["job_id"] = metadata.url.map(job_id)
metadata["id_ambiguous"] = metadata.job_id.duplicated(keep=False)
unique_ids = metadata[(metadata.job_id != "") & ~metadata.id_ambiguous]

audit = jobs.merge(metadata.drop(columns=["job_id", "id_ambiguous"]),
                   on="url", how="left", validate="one_to_one", indicator=True)
audit["job_id"] = audit.url.map(job_id)
audit["match_method"] = audit["_merge"].map({"both": "URL", "left_only": "chưa khớp"}).astype(str)
missing = audit.match_method.eq("chưa khớp")
by_id = audit.loc[missing, ["job_id"]].merge(
    unique_ids[["job_id", "sitemap_lastmod", "snapshot_year", "snapshot_period"]],
    on="job_id", how="left", validate="many_to_one",
)
matched_id = by_id.snapshot_year.notna().to_numpy()
audit.loc[missing, ["sitemap_lastmod", "snapshot_year", "snapshot_period"]] = (
    by_id[["sitemap_lastmod", "snapshot_year", "snapshot_period"]].to_numpy()
)
audit.loc[audit.index[missing][matched_id], "match_method"] = "ID duy nhất"
audit["posted_year"] = audit.posted_dt.dt.year.astype("Int64")
audit["same_as_lastmod"] = audit.posted_date.eq(audit.sitemap_lastmod) & audit.posted_date.ne("")
print("Cách nối tin chi tiết với sitemap:")
display(audit.match_method.value_counts().rename("so_url").to_frame())
print(f"Ngày ghi bằng lastmod (không chứng minh nguồn): {audit.same_as_lastmod.sum():,}")
print("Số URL gom được theo mốc sitemap:")
display(urls.groupby(["year", "snapshot_period"], dropna=False).size().rename("so_url").to_frame())
print("Năm ghi trong CSV chi tiết theo mốc sitemap của URL:")
display(pd.crosstab(audit.snapshot_year.fillna("không khớp"), audit.posted_year, dropna=False))


In [ ]:
requested_dates = {
    "07/2022": "2022-07-04", "12/2022": "2022-12-01",
    "03/2023": "2023-03-19", "06/2023": "2023-06-12",
    "09/2025": "2025-09-18",
}
url_check = urls.copy()
url_check["sitemap_lastmod"] = url_check[lastmod_col]
url_check["requested_date"] = url_check.snapshot_period.map(requested_dates)
url_check["lastmod_after_request"] = (
    url_check.requested_date.notna()
    & url_check.sitemap_lastmod.gt(url_check.requested_date.fillna(""))
)
print("lastmod muộn hơn mốc Wayback được yêu cầu (dấu hiệu cần xác minh capture thực):")
display(url_check[url_check.requested_date.notna()]
        .groupby("snapshot_period", sort=False)
        .agg(so_url=("url", "size"), sau_moc_yeu_cau=("lastmod_after_request", "sum")))


## 2. Phân bố ngày đang ghi trong CSV

Các năm **không được lấy với cùng mức độ phủ**: 2022 có hai mốc sitemap, 2023 có hai mốc, 2024 không có mốc riêng, 2025 có một mốc; 2026 quét sitemap live. Số tin giữa các năm không thể đọc như mức tăng giảm tuyển dụng của thị trường.


In [ ]:
valid = audit.dropna(subset=["posted_dt"]).copy()
monthly = valid.groupby(valid.posted_dt.dt.to_period("M")).size()
all_months = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M")
monthly = monthly.reindex(all_months, fill_value=0)
peak_month = monthly.idxmax()
pre_peak = monthly.loc[monthly.index < peak_month]

fig, axes = plt.subplots(2, 1, figsize=(15, 8), constrained_layout=True)
for ax, series, title in (
    (axes[0], monthly, "Số URL độc bản theo ngày ghi trong CSV — toàn giai đoạn"),
    (axes[1], pre_peak, f"Các tháng trước đỉnh {peak_month}"),
):
    ax.bar(range(len(series)), series.values, color="#3178a8", width=0.8)
    ax.set_title(title, loc="left")
    ax.set_ylabel("Số URL")
    ticks = [i for i, month in enumerate(series.index) if month.month in (1, 7)]
    ax.set_xticks(ticks, [str(series.index[i]) for i in ticks], rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].set_xlabel("Tháng")
plt.show()

print("Theo năm (URL độc bản):")
display(valid.groupby(valid.posted_dt.dt.year).size().rename("so_url").to_frame().T)
print("Các tháng nhiều bản ghi nhất:")
display(monthly.sort_values(ascending=False).head(12).rename("so_url").to_frame())


## 3. Vì sao năm 2022 cao?

Bảng dưới cho thấy mỗi tháng 2022 đến từ mốc sitemap nào. Một tin có ngày năm 2022 vẫn có thể được phát hiện trong sitemap yêu cầu năm 2023, vì sitemap chứa URL cũ. Đỉnh tháng 6/2022 tập trung quanh mốc sitemap tháng 7/2022. Mốc 2025 có `lastmod` muộn hơn ngày yêu cầu ở một số URL, nên cần kiểm tra capture thực tế trước khi tin rằng mọi sitemap trả về đúng ngày được yêu cầu.

Biểu đồ không phải mẫu ngẫu nhiên hay toàn bộ tin TopCV. Hãy dùng nó để mô tả **tập URL đã thu thập**, không suy ra quy mô thị trường theo năm.


In [ ]:
in_2022 = valid[valid.posted_dt.dt.year.eq(2022)].copy()
in_2022["month"] = in_2022.posted_dt.dt.to_period("M").astype(str)
print(f"Năm 2022: {len(in_2022):,} URL độc bản")
print("Tháng 6/2022 theo mốc sitemap:")
display(pd.crosstab(in_2022.month, in_2022.snapshot_period.fillna("không khớp")))
print("Ngày ghi trùng lastmod trong nhóm 2022 (không xác định nguồn gốc):",
      f"{in_2022.same_as_lastmod.sum():,} / {len(in_2022):,}")
